# Predict Engine Research

Notebook-first workflow for CatBoost regression. The primary artifact is `output/model.cbm`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root

In [ ]:
import json

import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold

from models.config import default_model_params
from models.regressor import build_regressor, resolve_categorical_columns, resolve_feature_columns
from runtime.device import get_device_report
from tracking.logger import get_logger
from tracking.wandb_tracker import finish_run, log_metrics, start_run, update_summary

In [ ]:
data_path = project_root / "data" / "raw" / "train.csv"
target_column = "expected_price"
feature_columns = None
categorical_columns = None

output_dir = project_root / "output"
model_path = output_dir / "model.cbm"
metrics_path = output_dir / "metrics.json"
feature_manifest_path = output_dir / "feature_manifest.json"

n_splits = 5
random_seed = 42

use_wandb = False
wandb_project = "predict_engine_research"
wandb_entity = None
wandb_run_name = None
wandb_tags = []

model_params = default_model_params()
model_params["random_seed"] = random_seed

{
    "data_path": str(data_path),
    "target_column": target_column,
    "output_dir": str(output_dir),
    "n_splits": n_splits,
    "model_params": model_params,
}

In [ ]:
logger = get_logger()
device_report = get_device_report()
logger.info("Torch device report: %s", device_report)
device_report

In [ ]:
frame = pd.read_csv(data_path)
feature_columns = resolve_feature_columns(frame, target_column, feature_columns)
categorical_columns = resolve_categorical_columns(frame, feature_columns, categorical_columns)

X = frame.loc[:, feature_columns].copy()
y = pd.to_numeric(frame[target_column], errors="raise")

logger.info("Loaded %s rows with %s features", len(frame), len(feature_columns))
logger.info("Detected categorical columns: %s", categorical_columns)
frame.head()

In [ ]:
run = start_run(
    enabled=use_wandb,
    project=wandb_project,
    entity=wandb_entity,
    name=wandb_run_name,
    tags=wandb_tags,
)
run

In [ ]:
splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_seed)
fold_results = []

for fold_index, (train_index, valid_index) in enumerate(splitter.split(X, y), start=1):
    logger.info("Starting fold %s/%s", fold_index, n_splits)

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]
    y_train = y.iloc[train_index]
    y_valid = y.iloc[valid_index]

    model = build_regressor(model_params)
    fit_kwargs = {}
    if categorical_columns:
        fit_kwargs["cat_features"] = categorical_columns

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        **fit_kwargs,
    )

    predictions = model.predict(X_valid)
    fold_metrics = {
        "fold": fold_index,
        "mae": float(mean_absolute_error(y_valid, predictions)),
        "rmse": float(root_mean_squared_error(y_valid, predictions)),
        "r2": float(r2_score(y_valid, predictions)),
    }
    fold_results.append(fold_metrics)

    logger.info("Fold %s metrics: %s", fold_index, fold_metrics)
    log_metrics(
        run,
        {
            "mae": fold_metrics["mae"],
            "rmse": fold_metrics["rmse"],
            "r2": fold_metrics["r2"],
        },
        step=fold_index,
    )

fold_results

In [ ]:
aggregate_metrics = {
    "mae": float(sum(item["mae"] for item in fold_results) / len(fold_results)),
    "rmse": float(sum(item["rmse"] for item in fold_results) / len(fold_results)),
    "r2": float(sum(item["r2"] for item in fold_results) / len(fold_results)),
}

logger.info("Aggregate CV metrics: %s", aggregate_metrics)
update_summary(run, aggregate_metrics)
aggregate_metrics

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

final_model = build_regressor(model_params)
final_fit_kwargs = {}
if categorical_columns:
    final_fit_kwargs["cat_features"] = categorical_columns

logger.info("Training final model on full dataset")
final_model.fit(X, y, **final_fit_kwargs)
final_model.save_model(model_path)

metrics_payload = {
    "fold_results": fold_results,
    "aggregate_metrics": aggregate_metrics,
    "model_path": str(model_path),
    "tree_count": int(final_model.tree_count_),
}
feature_manifest = {
    "target_column": target_column,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
}

metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
feature_manifest_path.write_text(json.dumps(feature_manifest, indent=2), encoding="utf-8")

update_summary(
    run,
    {
        "model_path": str(model_path),
        "tree_count": int(final_model.tree_count_),
    },
)
finish_run(run)

logger.info("Saved model to %s", model_path)
logger.info("Saved metrics to %s", metrics_path)
logger.info("Saved feature manifest to %s", feature_manifest_path)

{
    "model_path": str(model_path),
    "metrics_path": str(metrics_path),
    "feature_manifest_path": str(feature_manifest_path),
}